# Correlation of Metrics with SNS

In [1]:
import torch
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm.notebook import tqdm
from scipy.stats import spearmanr
from collections import defaultdict
from IPython.display import display

In [2]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [3]:
DATASET_NAMES = [
    "imagenet-100",
    "open-images-v7-50",
    "open-images-v7-30",
    "open-images-v7-10",
]
METRICS = [
    "test_MulticlassAccuracy",
    "test_MulticlassF1Score",
    "test_MulticlassPrecision",
    "test_MulticlassRecall",
]
MAIN_METRIC = "test_MulticlassAccuracy"
RBFN_RANGES = [
    "architecture",
    "rbfn_projection_head_num_kernels",
    "rbfn_projection_head_radial_function",
    "rbfn_projection_head_normalize",
]

In [4]:
RESULTS_PATH = Path("./results").resolve()
TRAINED_RBFN_MODELS_PATH_PREFIX = "./models/rbfn-ssl-projector-rbfn_"

In [5]:
def sns(centers: torch.Tensor, sigmas: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Scale-normalized separation (SNS) metric based on Gaussian interaction energy for RBF kernel geometry.

    Args:
        centers:    Tensor of shape (K, D)
        sigmas:     Tensor of shape (K,)
        eps:        Numerical stability constant

    Returns:
        Scalar tensor representing geometric quality
        (higher is better, max achieved near normalized distance = 1)
    """
    K, d = centers.shape
    dist = torch.cdist(centers, centers)
    scale = torch.sqrt(sigmas[:, None] ** 2 + sigmas[None, :] ** 2 + eps)
    delta = dist / (scale * (d**0.5))
    mask = ~torch.eye(K, dtype=torch.bool, device=delta.device)
    delta = delta[mask]
    return (delta**2 * torch.exp(-(delta**2))).mean()

In [6]:
for dataset_name in DATASET_NAMES:
    print("===== Dataset:", dataset_name, "=====")
    metric_by_run = defaultdict(dict)
    for metric_name, metric in [("SNS", sns)]:
        print("***** Metric:", metric_name, "*****")
        run_paths = [
            p
            for p in Path(TRAINED_RBFN_MODELS_PATH_PREFIX + dataset_name).iterdir()
            if p.is_dir()
        ]
        for run_path in tqdm(run_paths):
            trained_rbfn_ckpt_path = run_path / "version_0" / "checkpoints"

            # Find (single) checkpoint file for run
            trained_rbfn_ckpt = list(trained_rbfn_ckpt_path.rglob("*.ckpt"))
            assert (
                len(trained_rbfn_ckpt) == 1
            ), f"More than one checkpoint file found in '{trained_rbfn_ckpt}'."
            trained_rbfn_ckpt = trained_rbfn_ckpt[0]

            # Extract the state_dict from the loaded checkpoint
            trained_rbfn_state_dict = torch.load(trained_rbfn_ckpt)["state_dict"]

            # Extract projection head layers with centers and shapes from raw state_dict
            rbfn_layers_params = defaultdict(dict)
            for k, v in trained_rbfn_state_dict.items():
                if k.startswith("projection_head.layers.") and (
                    k.endswith(".kernels_centers") or k.endswith(".log_shapes")
                ):
                    layer_number, learned_param = k.split(".")[-2:]
                    rbfn_layers_params[int(layer_number)][learned_param] = v

            # Select the last layer of RBFN projection head with centers and shapes
            last_layer_number, last_rbfn_layer_params = next(
                reversed(sorted(rbfn_layers_params.items()))
            )

            centers = last_rbfn_layer_params["kernels_centers"].cpu()
            sigmas = (
                last_rbfn_layer_params["log_shapes"].cpu().exp()
            )  # NOTE: exp(log(sigmas)) -> sigmas

            metric_by_run[metric_name][int(run_path.name)] = metric(
                centers=centers, sigmas=sigmas
            ).item()

        metric_df = pd.DataFrame(
            list(metric_by_run[metric_name].items()), columns=["Name", metric_name]
        )

        # Read LogReg evaluation results
        RBFN_EVAL_RESULTS_PATH = RESULTS_PATH.joinpath(
            dataset_name, "rbfn", f"rbfn-ssl-projector-rbfn-eval_{dataset_name}.csv"
        )

        df_rbfn = pd.read_csv(RBFN_EVAL_RESULTS_PATH).merge(
            metric_df, on="Name", how="left"
        )

        pear_corr = defaultdict(list)
        for logreg_metric in METRICS:
            r_pear = df_rbfn[metric_name].corr(df_rbfn[logreg_metric])
            pear_corr["logreg_metric"].append(logreg_metric)
            pear_corr[f"r_pear({metric_name}, logreg_metric)"].append(r_pear)

        df_pear_corr = pd.DataFrame(pear_corr).round(2)
        display(df_pear_corr)

        spear_corr = defaultdict(list)
        for logreg_metric in METRICS:
            r_spear, p_spear = spearmanr(df_rbfn[metric_name], df_rbfn[logreg_metric])
            spear_corr["logreg_metric"].append(logreg_metric)
            spear_corr[f"r_spear({metric_name}, logreg_metric)"].append(r_spear)
            spear_corr["p_spear"].append(p_spear)

        df_spear_corr = pd.DataFrame(spear_corr).round(2)
        display(df_spear_corr)

===== Dataset: imagenet-100 =====
***** Metric: SNS *****


  0%|          | 0/216 [00:00<?, ?it/s]

,logreg_metric,"r_pear(SNS, logreg_metric)"
0,test_MulticlassAccuracy,0.87
1,test_MulticlassF1Score,0.87
2,test_MulticlassPrecision,0.85
3,test_MulticlassRecall,0.87


,logreg_metric,"r_spear(SNS, logreg_metric)",p_spear
0,test_MulticlassAccuracy,0.85,0.0
1,test_MulticlassF1Score,0.85,0.0
2,test_MulticlassPrecision,0.85,0.0
3,test_MulticlassRecall,0.85,0.0


===== Dataset: open-images-v7-50 =====
***** Metric: SNS *****


...

,logreg_metric,"r_spear(SNS, logreg_metric)",p_spear
0,test_MulticlassAccuracy,0.76,0.0
1,test_MulticlassF1Score,0.76,0.0
2,test_MulticlassPrecision,0.72,0.0
3,test_MulticlassRecall,0.76,0.0


===== Dataset: open-images-v7-10 =====
***** Metric: SNS *****


  0%|          | 0/216 [00:00<?, ?it/s]

,logreg_metric,"r_pear(SNS, logreg_metric)"
0,test_MulticlassAccuracy,0.81
1,test_MulticlassF1Score,0.81
2,test_MulticlassPrecision,0.79
3,test_MulticlassRecall,0.81


,logreg_metric,"r_spear(SNS, logreg_metric)",p_spear
0,test_MulticlassAccuracy,0.83,0.0
1,test_MulticlassF1Score,0.83,0.0
2,test_MulticlassPrecision,0.82,0.0
3,test_MulticlassRecall,0.83,0.0
